In [9]:
# %pip install statsmodels plotly ipywidgets yfinance


In [10]:
import numpy as np
import pandas as pd
import yfinance as yf
import statsmodels.api as sm
from statsmodels.tsa.stattools import adfuller


def fetch_prices(ticker_y: str, ticker_x: str, period: str) -> pd.DataFrame:
    """
    Downloads adjusted close prices for two tickers and returns a clean
    two-column DataFrame ['Y', 'X'] aligned on common trading dates.

    Fetches each ticker separately to avoid the MultiIndex column
    ambiguity that yf.download() introduces for multi-ticker requests
    (the column layout differs across yfinance versions/group_by modes).
    """
    frames = {}
    for ticker in (ticker_y, ticker_x):
        # auto_adjust=False keeps a dedicated 'Adj Close' column.
        # (Newer yfinance defaults to auto_adjust=True, which removes
        # 'Adj Close' and folds adjustment into 'Close' instead -- that
        # mismatch is what caused the original KeyError.)
        hist = yf.download(ticker, period=period, auto_adjust=False, progress=False)
        if hist.empty:
            raise ValueError(f"No data returned for '{ticker}'. Check the symbol (EGX tickers use the .CA suffix).")
        if isinstance(hist.columns, pd.MultiIndex):
            hist.columns = hist.columns.get_level_values(0)
        col = 'Adj Close' if 'Adj Close' in hist.columns else 'Close'
        frames[ticker] = hist[col]

    df = pd.concat(frames, axis=1).dropna()
    if df.empty:
        raise ValueError("No overlapping trading dates between the two tickers after alignment.")
    df.columns = ['Y', 'X']  # Y = ticker_y, X = ticker_x
    return df


def half_life_of_mean_reversion(spread: pd.Series) -> float:
    """
    Estimates the Ornstein-Uhlenbeck half-life of the spread by regressing
    day-over-day changes on the lagged level: d(spread) = theta * spread_lag + eps.
    Half-life = -ln(2) / theta. Returns np.nan if theta implies no mean reversion.
    """
    spread_lag = spread.shift(1).dropna()
    spread_ret = spread.diff().dropna()
    spread_lag = spread_lag.loc[spread_ret.index]

    X = sm.add_constant(spread_lag)
    model = sm.OLS(spread_ret, X).fit()
    theta = model.params.iloc[1]

    if theta >= 0:
        return np.nan  # no mean reversion implied
    return -np.log(2) / theta


def fetch_and_calculate_arbitrage(ticker_y: str, ticker_x: str, period: str = "2y", verbose: bool = True):
    """
    Fetches historical pricing from yfinance and calculates structural
    StatArb metrics (hedge ratio, spread, ADF cointegration test, half-life,
    rolling z-score) for the given pair. Pure function -- no hardcoded
    tickers, so it can be driven entirely from the picker widget below.
    """
    if verbose:
        print(f"[1/4] Ingesting historical market data for {ticker_y} and {ticker_x} via yfinance...")
    df = fetch_prices(ticker_y, ticker_x, period)
    if verbose:
        print(f"Loaded {len(df)} aligned daily bars.")

    df['log_y'] = np.log(df['Y'])
    df['log_x'] = np.log(df['X'])

    if verbose:
        print("[2/4] Computing static Ordinary Least Squares (OLS) hedge ratio...")
    Y_reg = df['log_y']
    X_reg = sm.add_constant(df['log_x'])
    model = sm.OLS(Y_reg, X_reg).fit()

    beta = model.params['log_x']
    alpha = model.params['const']
    df['spread'] = df['log_y'] - (beta * df['log_x'] + alpha)

    if verbose:
        print("[3/4] Running Augmented Dickey-Fuller (ADF) test for stationarity...")
    adf_result = adfuller(df['spread'], autolag='AIC')
    p_value = adf_result[1]
    crit_values = adf_result[4]

    if verbose:
        print("[4/4] Estimating mean-reversion half-life...")
    half_life = half_life_of_mean_reversion(df['spread'])
    correlation = df['log_y'].corr(df['log_x'])

    df['rolling_mean'] = df['spread'].rolling(window=20).mean()
    df['rolling_std'] = df['spread'].rolling(window=20).std()
    df['z_score'] = (df['spread'] - df['rolling_mean']) / df['rolling_std']

    stats = {
        'ticker_y': ticker_y, 'ticker_x': ticker_x, 'period': period,
        'beta': beta, 'alpha': alpha, 'correlation': correlation,
        'adf_stat': adf_result[0], 'p_value': p_value, 'crit_values': crit_values,
        'half_life': half_life, 'cointegrated': p_value < 0.05,
    }
    return df, stats


def print_stats(stats: dict):
    print("\n=============================================")
    print("      STATISTICAL ARBITRAGE ENGINE RESULTS    ")
    print("=============================================")
    print(f"Pair:                           {stats['ticker_y']} (Y) vs {stats['ticker_x']} (X)  |  period={stats['period']}")
    print(f"Log-price correlation:          {stats['correlation']:.4f}")
    print(f"Hedge Ratio (Elasticity Beta):  {stats['beta']:.4f}")
    print(f"Intercept Constant (Alpha):     {stats['alpha']:.4f}")
    print(f"ADF Statistic:                  {stats['adf_stat']:.4f}")
    print(f"ADF p-value:                    {stats['p_value']:.6f}")
    hl = stats['half_life']
    print(f"Half-life (mean reversion):     {hl:.1f} days" if pd.notna(hl)
            else "Half-life (mean reversion):     n/a (no mean reversion)")
    print("---------------------------------------------")
    print("Critical Values Thresholds:")
    for key, value in stats['crit_values'].items():
        print(f"  {key}: {value:.4f}")
    print("---------------------------------------------")
    print("RESULT: COINTEGRATION CONFIRMED" if stats['cointegrated'] else "RESULT: NO COINTEGRATION DETECTED")
    print("=============================================\n")


## Pick a Pair -- One Control Panel, Fully Interactive

Everything below is driven from this single panel: choose (or type) any two tickers and a lookback period, hit **Fetch & Analyze**, then drag the window/threshold/bin sliders and every chart, stat, and signal updates live -- no need to touch code or re-run cells to try a different pair.

In [11]:
import ipywidgets as widgets
from IPython.display import display, clear_output

EGX_TICKERS = [
    "TMGH.CA", "MASR.CA", "OCDI.CA", "COMI.CA", "HRHO.CA", "SWDY.CA",
    "ETEL.CA", "EAST.CA", "ABUK.CA", "AMOC.CA", "EFID.CA", "ORHD.CA",
    "ISPH.CA", "JUFO.CA", "EKHO.CA", "PHDC.CA", "ORAS.CA", "CIEB.CA",
]

ticker_y_box = widgets.Combobox(
    value="TMGH.CA", placeholder="e.g. TMGH.CA", options=EGX_TICKERS,
    description="Ticker Y", ensure_option=False,
    style={'description_width': '90px'}, layout=widgets.Layout(width='260px')
)
ticker_x_box = widgets.Combobox(
    value="MASR.CA", placeholder="e.g. MASR.CA", options=EGX_TICKERS,
    description="Ticker X", ensure_option=False,
    style={'description_width': '90px'}, layout=widgets.Layout(width='260px')
)
period_dd = widgets.Dropdown(
    options=["6mo", "1y", "2y", "3y", "5y", "max"], value="2y",
    description="Period", style={'description_width': '90px'},
    layout=widgets.Layout(width='220px')
)
run_button = widgets.Button(description="Fetch & Analyze", button_style="primary", icon="refresh")
status_html = widgets.HTML()

window_slider = widgets.IntSlider(value=20, min=5, max=60, step=1,
                                   description='Rolling window (days)',
                                   style={'description_width': '160px'},
                                   layout=widgets.Layout(width='420px'))
threshold_slider = widgets.FloatSlider(value=2.0, min=1.0, max=3.5, step=0.1,
                                        description='Z-score threshold',
                                        style={'description_width': '160px'},
                                        layout=widgets.Layout(width='420px'))
bins_slider = widgets.IntSlider(value=30, min=10, max=80, step=5,
                                 description='Histogram bins',
                                 style={'description_width': '160px'},
                                 layout=widgets.Layout(width='420px'))

dashboard_out = widgets.Output()

# Shared state: holds the currently loaded dataset/stats so slider tweaks
# can redraw instantly without re-hitting the network.
state = {"df": None, "stats": None}

top_row = widgets.HBox([ticker_y_box, ticker_x_box, period_dd, run_button])
slider_row = widgets.VBox([window_slider, threshold_slider, bins_slider])

display(widgets.VBox([top_row, status_html, slider_row, dashboard_out]))


def redraw(*_):
    if state["df"] is None:
        return
    with dashboard_out:
        clear_output(wait=True)
        print_stats(state["stats"])
        build_dashboard(state["df"], state["stats"],
                         window_slider.value, threshold_slider.value, bins_slider.value)


def on_run_clicked(_):
    y, x, period = ticker_y_box.value.strip(), ticker_x_box.value.strip(), period_dd.value
    status_html.value = f"<i>Fetching {y} vs {x} ({period})...</i>"
    try:
        df, stats = fetch_and_calculate_arbitrage(y, x, period, verbose=False)
        state["df"], state["stats"] = df, stats
        status_html.value = f"<b>Loaded:</b> {y} vs {x} | {period} | {len(df)} bars"
        redraw()
    except Exception as e:
        state["df"], state["stats"] = None, None
        status_html.value = f"<span style='color:crimson'><b>Error:</b> {e}</span>"
        with dashboard_out:
            clear_output(wait=True)

run_button.on_click(on_run_clicked)
# Sliders redraw instantly from the already-loaded data -- no refetch needed.
window_slider.observe(redraw, names='value')
threshold_slider.observe(redraw, names='value')
bins_slider.observe(redraw, names='value')


In [12]:
import plotly.graph_objects as go
from plotly.subplots import make_subplots


def build_dashboard(df: pd.DataFrame, stats: dict, window: int, z_threshold: float, bins: int):
    """Recomputes rolling stats for the chosen window/threshold and renders
    a linked, interactive Plotly dashboard (price performance, spread, z-score
    with live signals) plus a spread distribution histogram."""

    ticker_y, ticker_x = stats['ticker_y'], stats['ticker_x']
    d = df.copy()
    d['rolling_mean'] = d['spread'].rolling(window=window).mean()
    d['rolling_std'] = d['spread'].rolling(window=window).std()
    d['z_score'] = (d['spread'] - d['rolling_mean']) / d['rolling_std']

    normalized = d[['Y', 'X']] / d[['Y', 'X']].iloc[0] * 100
    long_signals = d[d['z_score'] < -z_threshold]
    short_signals = d[d['z_score'] > z_threshold]

    fig = make_subplots(
        rows=3, cols=1, shared_xaxes=True, vertical_spacing=0.06,
        row_heights=[0.32, 0.32, 0.36],
        subplot_titles=(
            f"Normalized Price Performance: {ticker_y} vs {ticker_x} (Start = 100)",
            "Log-Price Spread",
            f"Spread Z-Score (window={window}d, threshold=+/-{z_threshold})"
        )
    )

    fig.add_trace(go.Scatter(x=normalized.index, y=normalized['Y'], name=ticker_y,
                              line=dict(color='#2E86DE')), row=1, col=1)
    fig.add_trace(go.Scatter(x=normalized.index, y=normalized['X'], name=ticker_x,
                              line=dict(color='#EE5A24')), row=1, col=1)

    fig.add_trace(go.Scatter(x=d.index, y=d['spread'], name='Spread',
                              line=dict(color='#341f97')), row=2, col=1)
    fig.add_hline(y=d['spread'].mean(), line_dash='dash', line_color='red',
                   annotation_text='Mean', row=2, col=1)

    fig.add_trace(go.Scatter(x=d.index, y=d['z_score'], name='Z-Score',
                              line=dict(color='black')), row=3, col=1)
    fig.add_hline(y=z_threshold, line_dash='dash', line_color='crimson', row=3, col=1)
    fig.add_hline(y=-z_threshold, line_dash='dash', line_color='seagreen', row=3, col=1)
    fig.add_hline(y=0, line_dash='dot', line_color='gray', row=3, col=1)

    fig.add_trace(go.Scatter(x=short_signals.index, y=short_signals['z_score'],
                              mode='markers', name='Short Signal',
                              marker=dict(symbol='triangle-down', size=11, color='crimson')),
                  row=3, col=1)
    fig.add_trace(go.Scatter(x=long_signals.index, y=long_signals['z_score'],
                              mode='markers', name='Long Signal',
                              marker=dict(symbol='triangle-up', size=11, color='seagreen')),
                  row=3, col=1)

    fig.update_xaxes(rangeslider_visible=True, row=3, col=1)
    fig.update_layout(
        height=850, hovermode='x unified', template='plotly_white',
        legend=dict(orientation='h', yanchor='bottom', y=1.02, xanchor='right', x=1),
        margin=dict(t=60, b=40)
    )
    fig.show()

    hist_fig = go.Figure()
    hist_fig.add_trace(go.Histogram(x=d['spread'], nbinsx=bins, marker_color='#341f97'))
    hist_fig.add_vline(x=d['spread'].mean(), line_dash='dash', line_color='red',
                        annotation_text='Mean')
    hist_fig.update_layout(
        title='Distribution of Spread', xaxis_title='Spread', yaxis_title='Frequency',
        template='plotly_white', height=350, margin=dict(t=50, b=40)
    )
    hist_fig.show()

    half_life = half_life_of_mean_reversion(d['spread'].dropna())
    latest = d['z_score'].iloc[-1]
    if pd.notna(latest):
        if latest > z_threshold:
            signal = f"SHORT {ticker_y} / LONG {ticker_x} (spread rich)"
        elif latest < -z_threshold:
            signal = f"LONG {ticker_y} / SHORT {ticker_x} (spread cheap)"
        else:
            signal = "No active signal (within band)"
    else:
        signal = "n/a (not enough data for this window)"

    hl_text = f"{half_life:.1f} days" if pd.notna(half_life) else "n/a"
    display(widgets.HTML(
        f"<b>Latest z-score:</b> {latest:.2f} &nbsp;|&nbsp; "
        f"<b>Half-life:</b> {hl_text} &nbsp;|&nbsp; "
        f"<b>Signal:</b> {signal} &nbsp;|&nbsp; "
        f"<b>Signals in window:</b> {len(long_signals)} long / {len(short_signals)} short"
    ))


In [13]:
on_run_clicked(None)
